In [2]:
import pandas as pd
import numpy as np

train = pd.read_parquet('../data/raw/train_parsed.parquet')

print(f'Rows       : {len(train):,}')
print(f'SKUs       : {train["ItemCode"].nunique():,}')
print(f'Date range : {train["Date"].min().date()} → '
      f'{train["Date"].max().date()}')

Rows       : 711,980
SKUs       : 15,972
Date range : 2020-11-17 → 2025-09-05


In [3]:
daily = (
    train.groupby(['Date', 'ItemCode'])['Quantity']
    .sum()
    .clip(lower=0)
    .reset_index()
    .rename(columns={'Quantity': 'Qty'})
)
daily = daily[daily['Qty'] > 0]

all_dates = pd.date_range('2020-11-17', '2025-09-05', freq='D')
all_skus  = sorted(train['ItemCode'].unique())

grid = (
    daily.pivot_table(index='Date', columns='ItemCode',
                      values='Qty', fill_value=0)
    .reindex(index=all_dates, columns=all_skus, fill_value=0)
)

print(f'Grid shape : {grid.shape}')
print(f'Sparsity   : {(grid.values==0).mean():.1%}')

Grid shape : (1754, 15972)
Sparsity   : 98.3%


In [4]:
# Clean UnitPrice
train['Price'] = (
    train['UnitPrice']
    .astype(str)
    .str.replace(',', '', regex=False)
    .str.strip()
    .replace('', np.nan)
    .astype(float)
)

# Revenue per SKU
sku_revenue = (
    train[train['Quantity'] > 0]
    .assign(Revenue=lambda x: x['Quantity'] * x['Price'].fillna(0))
    .groupby('ItemCode')['Revenue']
    .sum()
    .reindex(all_skus, fill_value=0)
)

print('Top 10 SKUs by revenue:')
print(sku_revenue.sort_values(ascending=False).head(10))

Top 10 SKUs by revenue:
ItemCode
SKU-09458    7.437130e+13
SKU-08589    3.479892e+13
SKU-10120    2.067527e+13
SKU-11138    1.207705e+13
SKU-11131    9.617193e+12
SKU-10532    8.852585e+12
SKU-08063    7.802840e+12
SKU-07770    6.960696e+12
SKU-03456    6.875176e+12
SKU-08191    6.610502e+12
Name: Revenue, dtype: float64


In [5]:
# Sort by revenue descending, compute cumulative share
sku_rev_sorted = sku_revenue.sort_values(ascending=False)
cumshare       = sku_rev_sorted.cumsum() / sku_rev_sorted.sum()

abc = pd.Series('C', index=sku_rev_sorted.index)
abc[cumshare <= 0.80] = 'A'
abc[(cumshare > 0.80) & (cumshare <= 0.95)] = 'B'
# reindex back to original SKU order
abc = abc.reindex(all_skus)

print('ABC Distribution:')
print(abc.value_counts().sort_index())
print(f'\nA SKUs: {(abc=="A").sum()} — top 80% revenue')
print(f'B SKUs: {(abc=="B").sum()} — next 15% revenue')
print(f'C SKUs: {(abc=="C").sum()} — bottom 5% revenue')

ABC Distribution:
A       19
B       33
C    15920
Name: count, dtype: int64

A SKUs: 19 — top 80% revenue
B SKUs: 33 — next 15% revenue
C SKUs: 15920 — bottom 5% revenue


In [6]:
# Compute CV, zero_ratio, ADI per SKU
n_days = len(all_dates)

sku_mean       = grid.mean()
sku_std        = grid.std()
sku_cv         = (sku_std / sku_mean.replace(0, np.nan)).fillna(999)
sku_zero_ratio = (grid == 0).mean()
sku_days_sales = (grid > 0).sum()
sku_adi        = n_days / sku_days_sales.replace(0, np.nan)

xyz = pd.Series('Z', index=all_skus)
xyz[(sku_cv < 0.5)  & (sku_zero_ratio < 0.3)] = 'X'
xyz[(sku_cv < 1.0)  & (sku_zero_ratio < 0.6) &
    (xyz == 'Z')]                              = 'Y'

print('XYZ Distribution:')
print(xyz.value_counts().sort_index())
print(f'\nX SKUs : {(xyz=="X").sum()} — stable demand')
print(f'Y SKUs : {(xyz=="Y").sum()} — moderate variability')
print(f'Z SKUs : {(xyz=="Z").sum()} — intermittent/sparse')

XYZ Distribution:
Z    15972
Name: count, dtype: int64

X SKUs : 0 — stable demand
Y SKUs : 0 — moderate variability
Z SKUs : 15972 — intermittent/sparse


In [7]:
abcxyz = abc + xyz   # e.g. 'A' + 'X' = 'AX'

seg_counts = abcxyz.value_counts().sort_index()
print('ABC-XYZ Segment Distribution:')
print('='*35)
for seg, cnt in seg_counts.items():
    bar = '█' * (cnt // 100)
    print(f'  {seg} : {cnt:>5}  {bar}')

# Strategy per segment
strategy = {
    'AX': 'ML_model',
    'AY': 'ML_model',
    'AZ': 'ML_model',
    'BX': 'DOW_avg',
    'BY': 'DOW_avg',
    'BZ': 'median',
    'CX': 'mean',
    'CY': 'mean',
    'CZ': 'zero',
}
print('\nForecast strategy per segment:')
for seg, strat in strategy.items():
    cnt = seg_counts.get(seg, 0)
    print(f'  {seg} → {strat:<12} ({cnt} SKUs)')

ABC-XYZ Segment Distribution:
  AZ :    19  
  BZ :    33  
  CZ : 15920  ███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████

Forecast strategy per segment:
  AX → ML_model     (0 SKUs)
  AY → ML_model     (0 SKUs)
  AZ → ML_model     (19 SKUs)
  BX → DOW_avg      (0 SKUs)
  BY → DOW_avg      (0 SKUs)
  BZ → median       (33 SKUs)
  CX → mean         (0 SKUs)
  CY → mean         (0 SKUs)
  CZ → zero         (15920 SKUs)


In [8]:
sku_meta = pd.DataFrame({
    'sku'           : all_skus,
    'total_revenue' : sku_revenue.values,
    'rev_share'     : (sku_revenue / sku_revenue.sum()).values,
    'mean_qty'      : sku_mean.values,
    'std_qty'       : sku_std.values,
    'cv'            : sku_cv.values,
    'zero_ratio'    : sku_zero_ratio.values,
    'days_active'   : sku_days_sales.values,
    'adi'           : sku_adi.values,
    'abc'           : abc.values,
    'xyz'           : xyz.values,
    'abcxyz'        : abcxyz.values,
    'strategy'      : abcxyz.map(strategy).values,
}).set_index('sku')

print(sku_meta.head(10).to_string())
print(f'\nSaved metadata: {sku_meta.shape}')

           total_revenue     rev_share  mean_qty   std_qty         cv  zero_ratio  days_active         adi abc xyz abcxyz strategy
sku                                                                                                                               
SKU-00001   3.608433e+07  1.272363e-07  0.017104  0.228449  13.356627    0.991448           15  116.933333   C   Z     CZ     zero
SKU-00002   8.012686e+09  2.825339e-05  3.360319  4.831459   1.437798    0.489738          895    1.959777   C   Z     CZ     zero
SKU-00003   1.670068e+10  5.888798e-05  6.234322  7.297473   1.170532    0.395097         1061    1.653157   C   Z     CZ     zero
SKU-00004   8.361216e+08  2.948233e-06  0.375713  1.045296   2.782169    0.840935          279    6.286738   C   Z     CZ     zero
SKU-00005   2.242674e+09  7.907852e-06  0.627708  1.964298   3.129318    0.813569          327    5.363914   C   Z     CZ     zero
SKU-00006   3.380000e+05  1.191816e-09  0.002281  0.058459  25.634443    0.998290  

In [9]:
import os
os.makedirs('../data/processed', exist_ok=True)

grid.to_parquet('../data/processed/daily_sales.parquet')
sku_meta.to_parquet('../data/processed/sku_metadata.parquet')

print('Saved:')
print('  ../data/processed/daily_sales.parquet')
print('  ../data/processed/sku_metadata.parquet')
print()
print('='*50)
print('PREPROCESSING COMPLETE')
print('='*50)
print(f'Grid shape     : {grid.shape}')
print(f'SKU metadata   : {sku_meta.shape}')
print(f'ABC — A: {(abc=="A").sum()}  B: {(abc=="B").sum()}'
      f'  C: {(abc=="C").sum()}')
print(f'XYZ — X: {(xyz=="X").sum()}  Y: {(xyz=="Y").sum()}'
      f'  Z: {(xyz=="Z").sum()}')

Saved:
  ../data/processed/daily_sales.parquet
  ../data/processed/sku_metadata.parquet

PREPROCESSING COMPLETE
Grid shape     : (1754, 15972)
SKU metadata   : (15972, 12)
ABC — A: 19  B: 33  C: 15920
XYZ — X: 0  Y: 0  Z: 15972
